# Build the Hearing Corpus

| Item | Description |
|---|---|
| Goal | Add the transcripts to the hearing table and write the one table the summarization stage reads: one row per hearing with its metadata, its speaker-attributed turns, and the counts needed to decide what is usable. Nothing is filtered here; which hearings to drop is a sample decision for D1.4.1. |
| Input | `Interim/hearings.parquet` and `Interim/hearing_members.parquet` from D1.1.3; `Raw/<year>/html/<packageId>.htm` from D1.1.2. |
| Output | `Processed/hearings_all.parquet` in the D1.1 scratch folder: 36,378 rows, one per package, with the metadata columns of D1.1.3, the list column `turns`, the counts below, and `usable`. |
| Variables | • `turns`: list of speaking turns; each holds `bioGuideId`, `surname`, `honorific`, `is_member`, `section` (`spoken` or `qfr`), `n_paragraphs`, `n_qfr_paragraphs`, `n_words`, `text`<br>• `n_paragraphs`, `n_turns`, `n_chars`, `n_words`, `n_spoken_chars`: size of the transcript<br>• `n_front_matter`, `n_paragraphs_dropped`, `n_turns_unattributed`, `n_qfr_paragraphs`, `n_qfr_turns`: what was dropped, unattributed, or written rather than spoken<br>• `paragraph_rule`, `indent_share`: which paragraph rule ran and why<br>• `has_marker`, `has_text`: the two signals that a file holds a transcript<br>• `committee_ids`: every committee `authorityId` on the record, not only the first<br>• `usable`: `has_text` and `n_chars` above 8,650 |
| Run | Last run 2026-08-20, about ten minutes; the cell outputs shown are from that run. Not rerun: this file is the corpus every LLM stage read. It now sits in the Processed folder named above. |

## Step 1. Import packages

`BeautifulSoup` strips the HTML wrapper. `pyarrow` checks the written file.

In [ ]:
import re
import time
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from bs4 import BeautifulSoup

## Step 2. Define file locations

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D1 - Data Collection and Processing/D1.1 - GovInfo Congressional Hearings")
NOTEBOOK_NAME = "D1.1.4 - GovInfo - Build Hearing Corpus"
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"  # transcripts, written by D1.1.2
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"  # hearing tables, written by D1.1.3
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"

PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)

## Step 3. Load the hearing table and the member roster

The roster lookup is built once. Paragraphs with a dot leader are contents and index rows, not speech. `DROP_FRONT_MATTER` removes them, and the number dropped is recorded per hearing.

In [1]:
DROP_FRONT_MATTER = True

hearings = pd.read_parquet(INTERIM_FOLDER / "hearings.parquet")
members = pd.read_parquet(INTERIM_FOLDER / "hearing_members.parquet")

# packageId -> the member records of that hearing
roster = {package_id: group.to_dict("records") for package_id, group in members.groupby("packageId")}

print(f"hearings : {len(hearings):,}")
print(f"members  : {len(members):,} rows across {len(roster):,} hearings")
print(f"output   : {PROCESSED_FOLDER}")

hearings : 36,378
members  : 1,031,089 rows across 31,930 hearings
output   : /project2/njia_1089/acroh/projects/llm_method/data/raw/02_hearings


## Step 4. Define how a transcript is split into paragraphs

GPO serves a hearing as one block of fixed-width text. A paragraph starts at a four-space indent and runs to the next one, a rule that holds for 99.8 percent of the corpus. A few hearings have no indent, so the split falls back to blank lines when under 5 percent of lines are indented. Page stamps such as `[[Page 6]]` are removed and counted.

In [ ]:
INDENT = "    "  # GPO starts a paragraph with four spaces
INDENT_MIN_SHARE = 0.05  # below this share of indented lines the document does not use the convention
PAGE_MARKER = re.compile(r"\[\[Page[^\]]*\]\]")  # printed page stamps, written at column zero inside paragraphs


def extract_text(html):
    """Plain text of a GovInfo hearing page. BeautifulSoup copes with the stray unclosed tags in old renditions."""
    return BeautifulSoup(html, "html.parser").get_text()


def group_paragraphs(text):
    """Split hearing text into paragraphs. Returns the paragraphs, the rule that split them, and the indent share."""
    lines = [line for line in text.split("\n") if line.strip()]
    share = sum(1 for line in lines if line.startswith(INDENT)) / len(lines) if lines else 0.0
    rule = "indent" if not lines or share >= INDENT_MIN_SHARE else "blank_line"

    paragraphs = []
    current = []
    for line in text.split("\n"):
        line = PAGE_MARKER.sub("", line)
        stripped = line.strip()
        if rule == "indent":
            if not stripped:
                continue  # a blank line does not end a paragraph; only the next indent does
            if line.startswith(INDENT) and current:
                paragraphs.append(" ".join(current))
                current = []
            current.append(stripped)
        elif stripped:
            current.append(stripped)
        elif current:
            paragraphs.append(" ".join(current))
            current = []
    if current:
        paragraphs.append(" ".join(current))

    paragraphs = [re.sub(r"\s+", " ", p).strip() for p in paragraphs]
    if rule == "blank_line":
        paragraphs = [p for p in paragraphs if p]
    return paragraphs, rule, share

## Step 5. Define how a paragraph opens a speaking turn

GPO writes a change of speaker as an honorific, a surname, and a period at the start of a paragraph, as in `Mr. Fazio. Thank you.` A paragraph that does not open that way continues the current turn. A surname on the roster gets its `bioGuideId`, and the chair gets the sentinel `0000000`. Any other speaker, such as a witness, keeps a null id and the surname as written.

In [ ]:
ABBREV = r"(?:Mr|Mrs|Ms|Dr|Hon|Rev|Prof)"  # honorifics GPO writes with a period
TITLE = (r"(?:Miss|Senator|Representative|Congressman|Congresswoman|Chairman|Chairwoman|Chairperson|Secretary|General|"
         r"Admiral|Chief|Governor|Judge|Justice|Commissioner|Director|Colonel|Captain|Major|Lieutenant|Sergeant|"
         r"Professor|Ambassador|Mayor|Sheriff|Reverend|Rabbi|Delegate|Speaker|President)")  # honorifics written in full
SURNAME = r"[A-Z][A-Za-z'’\-]+(?:\s+[A-Z][A-Za-z'’\-]+)?"  # one or two capitalized words
# A paragraph opens a turn when it begins with an honorific and a surname, or "The Chairman", then an optional
# bracketed aside such as [continuing], then a period.
TURN_OPENING = re.compile(
    rf"^(?:(?P<chair>The\s+Chair(?:man|woman|person)?)"
    rf"|(?:{ABBREV}\.?\s+(?P<abbrev_name>{SURNAME}))"
    rf"|(?:(?P<title>{TITLE})\s+(?P<title_name>{SURNAME})))"
    rf"(?:\s*\[[^\]]*\])?\.{{1,2}}(?:\s|$)"
)
DOT_LEADER = re.compile(r"\.{3,}")  # a table-of-contents or index row, not speech
CHAIR_ID = "0000000"  # the chair is named by office, so there is no bioGuideId to give


def opening_of(paragraph):
    """The honorific and surname a paragraph opens with, or (None, None) when it continues the previous turn."""
    match = TURN_OPENING.match(paragraph)
    if not match:
        return None, None
    if match.group("chair"):
        return match.group("chair"), None
    name = match.group("abbrev_name") or match.group("title_name")
    honorific = match.group("title") or paragraph.split(None, 1)[0]
    return honorific, name.split()[-1] if name else None


def attribute(paragraphs, members, drop_front_matter=True):
    """Group paragraphs into speaking turns, each with the speaker's bioGuideId when the surname is on the roster."""
    by_surname = {}
    for member in members:
        name = member.get("name_lnf") or member.get("lastname")
        if name and member.get("bioGuideId"):
            by_surname.setdefault(name.split(",")[0].strip(), member["bioGuideId"])  # "Bennett, Robert F." -> "Bennett"

    turns = []
    current = None
    for paragraph in paragraphs:
        if drop_front_matter and DOT_LEADER.search(paragraph):
            continue
        honorific, surname = opening_of(paragraph)
        if honorific is None:
            if current is None:  # front matter before anyone speaks is kept as its own turn, not dropped
                current = {"bioGuideId": None, "surname": None, "honorific": None, "is_member": False, "paragraphs": []}
            current["paragraphs"].append(paragraph)
            continue
        if current is not None:
            turns.append(current)
        bio = CHAIR_ID if surname is None else by_surname.get(surname)
        current = {"bioGuideId": bio, "surname": surname, "honorific": honorific,
                   "is_member": bio is not None and bio != CHAIR_ID, "paragraphs": [paragraph]}
    if current is not None:
        turns.append(current)
    return turns

## Step 6. Define the written-question blocks and the usability test

Questions answered in writing are printed inside the hearing. A block opens with an aside such as `[Questions from Mr. Hoyer and responses follow:]` and closes at the next spoken turn. Each paragraph is labeled `spoken` or `qfr`, and nothing is dropped. A file holds a transcript unless it carries the `<TEXT NOT AVAILABLE>` marker, or is under 20 KB and opens no speaking turn.

In [ ]:
QFR_OPENING = re.compile(
    r"^\[?\s*(?:the following questions? (?:were|was) (?:not asked at the hearing|submitted)"
    r"|(?:a\s+)?questions?\s+from\b.{0,90}?(?:responses?|answers?)\s+follows?"
    r"|questions?\s+submitted\s+(?:by|to|for the record)"
    r"|the response to the aforementioned question follows)",
    re.I,
)
NO_TEXT_MARKER = b"<TEXT NOT AVAILABLE>"  # GovInfo writes this where the body of an undigitized hearing would be
COVER_PAGE_MAX_BYTES = 20_000  # a cover page is a few KB; a hearing with proceedings is 100 KB or more


def label_sections(paragraphs):
    """One label per paragraph: "spoken", or "qfr" inside a block of written questions."""
    labels = []
    in_block = False
    for paragraph in paragraphs:
        if QFR_OPENING.match(paragraph):
            in_block = True
        elif in_block and TURN_OPENING.match(paragraph):
            in_block = False
        labels.append("qfr" if in_block else "spoken")
    return labels


def has_usable_text(path):
    """Does the HTML file hold a hearing, or only its cover page?"""
    path = Path(path)
    if not path.exists():
        return {"exists": False, "bytes": 0, "n_turns": 0, "has_marker": False, "has_text": False}
    raw = path.read_bytes()
    if NO_TEXT_MARKER in raw:
        return {"exists": True, "bytes": len(raw), "n_turns": 0, "has_marker": True, "has_text": False}
    if len(raw) > COVER_PAGE_MAX_BYTES:  # too big to be a cover page; the turn count is skipped
        return {"exists": True, "bytes": len(raw), "n_turns": -1, "has_marker": False, "has_text": True}
    paragraphs, _, _ = group_paragraphs(extract_text(raw))
    n_turns = sum(1 for p in paragraphs if TURN_OPENING.match(p))
    return {"exists": True, "bytes": len(raw), "n_turns": n_turns, "has_marker": False, "has_text": n_turns > 0}


def parse_transcript(path):
    """One HTML file to its paragraphs, their section labels, and the counts that describe them."""
    text = extract_text(Path(path).read_bytes())
    paragraphs, rule, share = group_paragraphs(text)
    sections = label_sections(paragraphs)
    n_blocks = sum(1 for i, label in enumerate(sections) if label == "qfr" and (i == 0 or sections[i - 1] == "spoken"))
    return {
        "paragraphs": paragraphs,
        "sections": sections,
        "paragraph_rule": rule,
        "indent_share": share,
        "n_paragraphs": len(paragraphs),
        "n_chars": sum(len(p) for p in paragraphs),
        "n_front_matter": sum(1 for p in paragraphs if DOT_LEADER.search(p)),
        "n_page_markers": len(PAGE_MARKER.findall(text)),
        "n_qfr_blocks": n_blocks,
        "n_qfr_paragraphs": sections.count("qfr"),
        "n_spoken_chars": sum(len(p) for p, s in zip(paragraphs, sections) if s == "spoken"),
    }

## Step 7. Define one hearing

`build_row` parses one transcript, groups its paragraphs into turns, and returns the counts. Written questions arrive without a speaker opening and land inside the open turn. A turn is therefore labeled `qfr` when any of its paragraphs is. A package without a file returns the empty shape.

In [ ]:
def turn_sections(paragraphs, sections, turns):
    """Per turn, its section label and how many of its paragraphs are qfr, by walking the paragraph list forward."""
    out = []
    cursor = 0
    for turn in turns:
        labels = []
        for wanted in turn["paragraphs"]:
            while cursor < len(paragraphs):
                if paragraphs[cursor] == wanted:
                    labels.append(sections[cursor])
                    cursor += 1
                    break
                cursor += 1
            else:
                labels.append("spoken")
        n_qfr = sum(1 for value in labels if value == "qfr")
        out.append(("qfr" if n_qfr else "spoken", n_qfr))
    return out


def build_row(package_id, year):
    """Transcript-derived fields for one package, or the empty shape when the file is absent or empty."""
    path = RAW_FOLDER / str(year) / "html" / f"{package_id}.htm"
    status = has_usable_text(path)

    empty = {
        "turns": [],
        "n_paragraphs": 0, "n_turns": 0, "n_chars": 0, "n_words": 0,
        "n_front_matter": 0, "n_qfr_paragraphs": 0, "n_qfr_turns": 0,
        "n_spoken_chars": 0, "n_paragraphs_dropped": 0,
        "n_turns_unattributed": 0,
        "paragraph_rule": None, "indent_share": 0.0,
        "html_exists": bool(status["exists"]),
        "html_bytes": int(status["bytes"]),
        "has_marker": bool(status["has_marker"]),
        "has_text": bool(status["has_text"]),
    }
    if not status["exists"]:
        return empty

    parsed = parse_transcript(path)
    paragraphs = parsed["paragraphs"]
    if not paragraphs:
        return empty

    grouped = attribute(paragraphs, roster.get(package_id, []), drop_front_matter=DROP_FRONT_MATTER)
    labels = turn_sections(paragraphs, parsed["sections"], grouped)

    turns = []
    kept = 0
    for turn, (label, n_qfr) in zip(grouped, labels):
        kept += len(turn["paragraphs"])
        turns.append({
            "bioGuideId": turn["bioGuideId"],
            "surname": turn["surname"],
            "honorific": turn["honorific"],
            "is_member": bool(turn["is_member"]),
            "section": label,
            "n_paragraphs": len(turn["paragraphs"]),
            "n_qfr_paragraphs": n_qfr,
            "n_words": sum(len(p.split()) for p in turn["paragraphs"]),
            "text": "\n\n".join(turn["paragraphs"]),
        })

    return {
        **empty,
        "turns": turns,
        "n_paragraphs": parsed["n_paragraphs"],
        "n_turns": len(turns),
        "n_chars": parsed["n_chars"],
        "n_words": sum(t["n_words"] for t in turns),
        "n_front_matter": parsed["n_front_matter"],
        "n_qfr_paragraphs": parsed["n_qfr_paragraphs"],
        "n_qfr_turns": sum(1 for t in turns if t["section"] == "qfr"),
        "n_spoken_chars": parsed["n_spoken_chars"],
        "paragraph_rule": parsed["paragraph_rule"],
        "indent_share": parsed["indent_share"],
        "n_paragraphs_dropped": parsed["n_paragraphs"] - kept,
        "n_turns_unattributed": sum(1 for t in turns if t["bioGuideId"] is None),
    }

## Step 8. Walk the corpus

About 18 milliseconds per transcript, so the full pass takes about ten minutes in one process.

In [3]:
started = time.time()
rows = []
for n, hearing in enumerate(hearings.itertuples(), 1):
    rows.append({"packageId": hearing.packageId, **build_row(hearing.packageId, hearing.year)})
    if n % 5000 == 0:
        rate = n / (time.time() - started)
        print(f"  {n:,}/{len(hearings):,}  {rate:.0f} hearings/s  eta {(len(hearings) - n) / rate / 60:.1f} min")

text = pd.DataFrame(rows)
elapsed = time.time() - started
print(f"\nparsed {len(text):,} packages in {elapsed / 60:.1f} min")
print(f"  with an HTML file      {int(text['html_exists'].sum()):>7,}")
print(f"  with text in it        {int(text['has_text'].sum()):>7,}")
print(f"  carrying the marker    {int(text['has_marker'].sum()):>7,}")
print(f"  total turns            {int(text['n_turns'].sum()):>7,}")
print(f"  total words            {int(text['n_words'].sum()):>7,}")

  5,000/36,378  48 hearings/s  eta 10.8 min


  10,000/36,378  51 hearings/s  eta 8.6 min


  15,000/36,378  54 hearings/s  eta 6.6 min


  20,000/36,378  55 hearings/s  eta 5.0 min


  25,000/36,378  56 hearings/s  eta 3.4 min


  30,000/36,378  58 hearings/s  eta 1.8 min


  35,000/36,378  58 hearings/s  eta 0.4 min



parsed 36,378 packages in 10.3 min
  with an HTML file       35,503
  with text in it         33,363
  carrying the marker        378
  total turns            7,570,429
  total words            1,160,084,863


## Step 9. Join to the metadata, mark the usable rows, and write

`committee_ids` lists every committee on the record, where the scalar `committee_id` holds only the first. A row is usable when `has_text` is true and the text exceeds 8,650 characters, the length rule inherited from the earlier pipeline. The two signals disagreed on 5 of 2,500 sampled hearings, so both must hold. Arrow cannot read a nested text column above 2 GB in one row group, so the file uses groups of 2,000 rows.

In [4]:
META = ["packageId", "title", "congress", "session", "chamber", "branch",
        "docClass", "dateIssued", "year", "heldDates", "heldDate_first",
        "committees", "committee_id", "committee_name", "n_committees",
        "members", "witnesses", "n_members", "n_members_with_bioguide",
        "n_witnesses", "has_transcript"]

corpus = hearings[META].merge(text, on="packageId", how="left", validate="one_to_one")

# Every committee on the record. 53 packages carry a committee with a null authorityId; those entries are left out.
corpus["committee_ids"] = [[c["authorityId"] for c in v if c["authorityId"]] if v is not None else []
                           for v in corpus["committees"]]
corpus = corpus.sort_values(["year", "packageId"]).reset_index(drop=True)

hidden = int((corpus["committee_ids"].map(len) > 1).sum())
print(f"committee_ids built; {hidden:,} packages list more than one committee")

# The inherited length rule: the earlier pipeline kept a hearing when the characters of its parsed paragraphs
# exceeded 8,650, a number its author had checked by eye. n_chars is that same sum.
MIN_CHARS = 8650
corpus["usable"] = corpus["has_text"] & (corpus["n_chars"] > MIN_CHARS)

n_text = int(corpus["has_text"].sum())
n_len = int((corpus["n_chars"] > MIN_CHARS).sum())
print(f"\nusability, on {len(corpus):,} packages:")
print(f"  has_text                     {n_text:>7,}")
print(f"  n_chars > {MIN_CHARS}              {n_len:>7,}")
print(f"  usable (both)                {int(corpus['usable'].sum()):>7,}  ({corpus['usable'].mean():.1%})")
print(f"  dropped by has_text only     {int((~corpus['has_text'] & (corpus['n_chars'] > MIN_CHARS)).sum()):>7,}")
print(f"  dropped by length only       {int((corpus['has_text'] & (corpus['n_chars'] <= MIN_CHARS)).sum()):>7,}")

target = PROCESSED_FOLDER / "hearings_all.parquet"
corpus.to_parquet(target, compression="zstd", index=False, row_group_size=2000)

size = target.stat().st_size
groups = pq.ParquetFile(target).metadata.num_row_groups
print(f"wrote {target}")
print(f"  {len(corpus):,} rows in {groups} row groups, {size / 1e6:,.0f} MB zstd")
print(f"  {corpus['n_words'].sum() / 1e6:,.0f} million words across {int(corpus['n_turns'].sum()):,} turns")
print(f"  columns: {len(corpus.columns)}")

committee_ids built; 763 packages list more than one committee

usability, on 36,378 packages:
  has_text                      33,363
  n_chars > 8650               33,349
  usable (both)                 33,324  (91.6%)
  dropped by has_text only          25
  dropped by length only            39


wrote /project2/njia_1089/acroh/projects/llm_method/data/raw/02_hearings/hearings_all.parquet
  36,378 rows in 19 row groups, 2,504 MB zstd
  1,160 million words across 7,570,429 turns
  columns: 40


## Step 10. Check the written file

Eight checks on the file read back from disk. The first five stop the notebook; the last three describe the corpus.

- One row per package, `packageId` unique.
- No row carries turns without an HTML file.
- `n_turns` matches the stored turn list on every row.
- `n_words` recomputed from the turns matches on 2,000 sampled rows.
- No row attributes more written-question paragraphs than were parsed.
- Which paragraph rule ran, and what the fallback recovered.
- Usable coverage by year; 1995 and 1996 should be near zero.
- The committee identifiers survived the round trip, counted from `committee_ids`.

In [5]:
back = pd.read_parquet(target)  # reading the whole file, nested column included, is itself check zero

assert len(back) == len(hearings), f"{len(back):,} rows against {len(hearings):,} packages"
assert not back["packageId"].duplicated().any(), "duplicate packageId"
print(f"1) {len(back):,} rows, one per package, packageId unique")

ghost = back.loc[~back["html_exists"] & (back["n_turns"] > 0)]
silent = back.loc[back["html_exists"] & back["has_text"] & (back["n_turns"] == 0)]
assert len(ghost) == 0, f"{len(ghost)} rows have turns but no HTML file"
print(f"2) no rows carry text without a file; {len(silent):,} have a file with text but produced no turn")

bad = back.loc[back["turns"].map(len) != back["n_turns"]]
assert len(bad) == 0, f"{len(bad)} rows where n_turns disagrees with the turn list"
print("3) n_turns matches the stored turn list on every row")

check = back.loc[back["n_turns"] > 0].head(2000)
recomputed = [sum(t["n_words"] for t in turns) for turns in check["turns"]]
assert (pd.Series(recomputed, index=check.index) == check["n_words"]).all(), "n_words does not match the turns"
print(f"4) n_words recomputed from turns matches on {len(check):,} sampled rows")

qfr = back.loc[back["n_qfr_paragraphs"] > 0]
attributed = pd.Series([sum(t["n_qfr_paragraphs"] for t in turns) for turns in qfr["turns"]], index=qfr.index)
over = qfr.loc[attributed > qfr["n_qfr_paragraphs"]]
assert len(over) == 0, f"{len(over)} rows attribute more QFR paragraphs than were parsed"
print(f"5) {len(qfr):,} hearings carry questions for the record ({int(back['n_qfr_turns'].sum()):,} turns); none over-attributed")

print("\n6) paragraph rule, and what the fallback recovered:")
rules = back.loc[back["html_exists"], "paragraph_rule"].value_counts(dropna=False)
print("   " + rules.to_string().replace("\n", "\n   "))
fallback = back.loc[back["paragraph_rule"] == "blank_line"]
if len(fallback):
    print(f"   fallback fired on {len(fallback):,} packages; median {fallback['n_paragraphs'].median():,.0f} "
          f"paragraphs and {fallback['n_turns'].median():,.0f} turns each")
    stuck = back.loc[back["usable"] & (back["n_chars"] / back["n_paragraphs"].clip(lower=1) > 2000)]
    print(f"   usable rows still averaging >2,000 chars per paragraph: {len(stuck):,}")

print("\n7) usable coverage by year, 1995-96 should be near zero:")
cov = back.groupby("year").agg(hearings=("packageId", "size"), usable=("usable", "sum"), median_words=("n_words", "median"))
cov["pct"] = (cov["usable"] / cov["hearings"]).map("{:.0%}".format)
print(cov.to_string())

print("\n8) committee identifiers survived the round trip:")
for cid, label in [("hssy00", "House Science (all three names)"), ("sscm00", "Senate Commerce, Science, and Transportation")]:
    hit = pd.Series([cid in v for v in back["committee_ids"]], index=back.index)
    scalar = int(back["committee_id"].eq(cid).sum())
    print(f"   {cid}  {int(hit.sum()):>5,} packages, {int(back.loc[hit, 'usable'].sum()):,} usable   {label}")
    print(f"            the primary-committee column alone would find {scalar:,}")
both = pd.Series([{"hssy00", "sscm00"}.issubset(set(v)) for v in back["committee_ids"]], index=back.index)
print(f"   both       {int(both.sum()):>5,} packages carry both committees")
print("\n   No filtering was applied. The next step decides what to drop.")

1) 36,378 rows, one per package, packageId unique
2) no rows carry text without a file; 0 have a file with text but produced no turn
3) n_turns matches the stored turn list on every row
4) n_words recomputed from turns matches on 2,000 sampled rows
5) 3,855 hearings carry questions for the record (13,139 turns); none over-attributed

6) paragraph rule, and what the fallback recovered:
   paragraph_rule
   indent        35381
   blank_line      122
   fallback fired on 122 packages; median 164 paragraphs and 4 turns each
   usable rows still averaging >2,000 chars per paragraph: 49

7) usable coverage by year — 1995-96 should be near zero:
      hearings  usable  median_words  pct
year                                     
1995       253       2         106.0   1%
1996       141       2         111.0   1%
1997       456     354       30228.5  78%
1998       400     280       27665.0  70%
1999       850     734       28659.5  86%
2000       837     762       27893.0  91%
2001      1125   

   hssy00    964 packages, 941 usable   House Science (all three names)
            the primary-committee column alone would find 940
   sscm00  1,420 packages, 1,413 usable   Senate Commerce, Science, and Transportation
            the primary-committee column alone would find 1,414
   both           2 packages carry both committees

   No filtering was applied. The next step decides what to drop.
